In [ ]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import requests
import pandas as pd
import numpy as np
from time import sleep
import os

# CONFIG 
OUTPUT_DIR = "domestic_commodity_standardized_onebyone"
os.makedirs(OUTPUT_DIR, exist_ok=True)

HARDCODED_RATE_CURRENCIES = {"BYR", "TMT"}

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.


. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.


. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.



# SERIES BUILDER

FINAL_COLS = [
    "date", "price_usd", "price_local", "currency", "price_per_unit", "unit_std",
    "commodity_name", "iso3_country_code", "country", "market",
    "price_type", "unit", "price_source", "fill_method"
]


def build_series_df(datapoints, meta: dict, price_source: str) -> pd.DataFrame | None:
    rows = []
    for dp in datapoints:
        price_usd   = dp.get("price_value_dollar")
        price_local = dp.get("price_value")
        if price_local is None:
            price_local = dp.get("price_value_nominal")

        if price_usd is None and price_local is None:
            continue

        rows.append({
            "date":              dp["date"],
            "price_usd":         price_usd,
            "price_local":       price_local,
            "currency":          meta["currency"],
            "price_per_unit":    compute_price_per_unit(price_usd, meta["unit"]),
            "unit_std":          get_unit_std(meta["unit"]),
            "commodity_name":    meta["commodity_name"],
            "iso3_country_code": meta["iso3"],
            "country":           meta["country"],
            "market":            meta["market"],
            "price_type":        meta["price_type"],
            "unit":              meta["unit"],
            "price_source":      price_source,
            "fill_method":       "original",
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()

    df = (df.groupby("date", as_index=False)
            .apply(lambda g: g.loc[g.isna().sum(axis=1).idxmin()], include_groups=False)
            .reset_index(drop=True))

    full_range = pd.date_range(start=df["date"].min(),
                               end=df["date"].max(), freq="MS")
    df = (df.set_index("date")
            .reindex(full_range)
            .rename_axis("date")
            .reset_index())

    for col in ["commodity_name", "iso3_country_code", "country", "market",
                "price_type", "unit", "unit_std", "currency",
                "price_source", "fill_method"]:
        df[col] = df[col].ffill().bfill()

    df["fill_method"] = df["fill_method"].fillna("original")
    return df


# DOMESTIC SERIES

print("=" * 60)
print("FETCHING DOMESTIC SERIES")
print("=" * 60)

url_dom  = "https://fpma.fao.org/giews/v4/global/price_module/api/v1/FpmaSerieDomestic/?newerThan=12 months ago"
dom_data = requests.get(url_dom).json()
print(f"Total domestic series: {len(dom_data['results'])}")

commodity_buckets: dict[str, list[pd.DataFrame]] = {}

fill_stats = {
    "original":           0,
    "implicit_rate":      0,
    "hardcoded_rate_tmt": 0,
    "hardcoded_rate_byr": 0,
    "unfilled":           0,
}

for item in dom_data["results"]:
    uuid           = item["uuid"]
    commodity_name = item.get("commodity_name", "Unknown")
    country        = item.get("country_name", "Unknown")
    iso3           = item.get("iso3_country_code", "Unknown")
    market         = item.get("market_name", "Unknown")
    price_type     = item.get("price_type", "Unknown")
    unit           = item.get("measure_unit_label", "Unknown")
    currency       = item.get("currency", "Unknown")

    url_price = (
        f"https://fpma.fao.org/giews/v4/global/price_module/api/v1/"
        f"FpmaSeriePrice/?uuid__in={uuid}&periodicity=monthly"
    )
    resp = requests.get(url_price).json()
    if resp["count"] == 0:
        sleep(0.2)
        continue

    datapoints = resp["results"][0]["datapoints"]
    meta = dict(commodity_name=commodity_name, iso3=iso3, country=country,
                market=market, price_type=price_type, unit=unit, currency=currency)

    df = build_series_df(datapoints, meta, price_source="Domestic")
    if df is None:
        sleep(0.2)
        continue

    if df["price_usd"].isna().all() and df["price_local"].isna().all():
        print(f"  [SKIP] All-null series: {commodity_name} | {market}, {country}")
        sleep(0.2)
        continue

    missing_before = df["price_usd"].isna().sum()
    if missing_before > 0:
        df, n_filled = fill_usd(df, currency)
    else:
        n_filled = 0

    df = recompute_price_per_unit(df)

    missing_after = df["price_usd"].isna().sum()
    fill_stats["unfilled"] += missing_after
    if n_filled > 0:
        method = (df.loc[df["fill_method"] != "original", "fill_method"]
                    .iloc[0]
                  if (df["fill_method"] != "original").any()
                  else "unknown")
        fill_stats[method] = fill_stats.get(method, 0) + n_filled
    else:
        fill_stats["original"] += (missing_before - missing_after)

    print(f"  {commodity_name} | {market}, {country} | "
          f"{len(df)} rows | filled {n_filled}/{missing_before} missing USD")

    commodity_buckets.setdefault(commodity_name, []).append(df[FINAL_COLS])
    sleep(0.2)


# SAVE 

print("\n" + "=" * 60)
print("SAVING — one CSV per commodity")
print("=" * 60)

import warnings

saved_count = 0
for commodity_name, frames in commodity_buckets.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        combined = pd.concat(frames, ignore_index=True)

    combined = combined.sort_values(
        ["country", "market", "date"]
    ).reset_index(drop=True)

    filename = f"{sanitize_filename(commodity_name)}.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    combined.to_csv(filepath, index=False)
    saved_count += 1
    print(f"  Saved: {filepath}  ({len(combined):,} rows)")

print(f"\nTotal commodities saved : {saved_count}")
print(f"Output directory        : {OUTPUT_DIR}/")

print(f"\nFill strategy stats:")
for method, count in fill_stats.items():
    print(f"  {method}: {count:,}")

# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝